# ShelfSense: end-to-end notebook

Cleaning → exploration → modelling → evaluation. Runs in Google Colab or locally.

**Before running:** set `GITHUB_USER` below, then upload the raw CSV when prompted (download link is in `data/README.md`).

In [ ]:
GITHUB_USER = 'AsadAli323'
import os, sys
if not os.path.exists('amz-intelligence'):
    !git clone https://github.com/{GITHUB_USER}/amz-intelligence.git
%cd amz-intelligence
!pip install -q -r requirements.txt
sys.path.insert(0, 'src')

## 1. Load the raw data

In [ ]:
from google.colab import files  # skip this cell if running locally
up = files.upload()  # choose amazon_sales_data_uncleaned.csv
RAW = list(up)[0]

In [ ]:
import pandas as pd, numpy as np
raw = pd.read_csv(RAW)
print(raw.shape)
raw.head(3)

## 2. Clean
The raw export stores numbers as text, repeats products across scrape runs and has no category.

In [ ]:
from shelfsense.cleaning import clean
df = clean(raw)
print(f'{len(raw):,} raw rows -> {len(df):,} unique products')
df.isna().mean().round(3).sort_values(ascending=False).head(8)

## 3. Explore

In [ ]:
import matplotlib.pyplot as plt
df['category'].value_counts().sort_values().plot.barh(figsize=(7,5), title='Products per category'); plt.show()
np.log10(df['price'].dropna()).plot.hist(bins=40, title='log10(price)'); plt.show()

In [ ]:
s = df.dropna(subset=['units_sold_month'])
print(s[['units_sold_month','reviews','rating','price','discount_pct']].corr(method='spearman').round(2)['units_sold_month'])
print(s.groupby('is_sponsored')['units_sold_month'].median())

## 4. Train and compare models
* **Demand tier** (Low ≤100 / Medium 200-999 / High ≥1000 units per month): Logistic Regression, Random Forest, LightGBM
* **Price** (log target): Ridge, LightGBM

In [ ]:
from shelfsense.train import add_features, train_demand, train_price
feat = add_features(df)
d_model, d_best, d_res, d_base, d_n = train_demand(feat)
print('Majority baseline accuracy:', d_base)
pd.DataFrame(d_res).T

In [ ]:
p_model, p_best, p_res, p_base, p_n = train_price(feat)
print('Median-price baseline MAE: $', p_base)
pd.DataFrame(p_res).T

## 5. Try a prediction

In [ ]:
row = pd.DataFrame([{'title': 'Sony WH-1000XM5 Wireless Noise Cancelling Headphones', 'price': 328.0,
  'rating': 4.6, 'reviews': 9000, 'discount_pct': 12, 'is_sponsored': 0, 'is_best_seller': 0,
  'is_amazons_choice': 0, 'is_limited_deal': 0, 'has_coupon': 0, 'has_sustainability_badge': 0,
  'is_renewed': 0, 'title_words': 7, 'category': 'Headphones & Audio'}])
X = add_features(row)
print(dict(zip(d_model.classes_, d_model.predict_proba(X)[0].round(2))))

## 6. Retrain on the full data, save, and download
This writes `models/*.joblib` and `reports/metrics.json` and regenerates the README figures.

In [ ]:
from shelfsense import train
train.main(RAW, out='models', reports='reports')
!python reports/make_figures.py {RAW}

In [ ]:
!zip -qr trained_outputs.zip models reports
files.download('trained_outputs.zip')  # unzip into your repo folder, then commit and push